# Step 2 — Time-Series Network Analysis and Shared Connection Point

From this step onward the project correction is applied: **S1, S2 and S3 are CLOSED for all analysis**, and the original CIGRE loads are halved. The four company profiles are treated as one logistics area's aggregate electrical connection for siting, while remaining separate load elements for traceability.

### What this cell does — Imports and inputs

Loads the 336-hour active/reactive depot profiles and the chosen shared bus. No EV charger design is needed in Step 2.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data import load_base_loads, load_bus_mapping, shared_bus
from src.network.model import build_network
from src.network.siting import screen_shared_connection_points
from src.network.timeseries import zero_schedule, run_timeseries, network_summary, build_snapshot_network
from src.network.contingency import run_n1, SWITCH_CONFIGS_ALL_CLOSED
from src.reporting.plots import save_network_plot

base_p, base_q = load_base_loads()
bus_map = load_bus_mapping(require_shared=True)
selected_bus = shared_bus(bus_map)
network_factory = lambda: build_network(halve_existing_loads=True, close_ring_switches=True)
print("Selected shared logistics-area bus:", selected_bus)


### What this cell does — Shared-bus screening

Aggregates all four supplied depot base-load profiles at the hour of maximum aggregate demand and tests that complete logistics-area load at each candidate MV bus. The ranking is electrical evidence for the connection-point choice; it does not pretend the CIGRE buses are a geographical map.

In [ ]:
screening = screen_shared_connection_points(network_factory, base_p, base_q)
config.NETWORK_DIR.mkdir(parents=True, exist_ok=True)
screening.to_csv(config.BUS_SCREENING, index=False)
print("Chosen Bus", selected_bus, "screening row:")
display(screening.loc[screening.candidate_bus == selected_bus])
print("Top candidates:")
display(screening.head(8))


### What this cell does — 336-hour base-load simulation

Runs the half-loaded, all-closed CIGRE network for all 336 hours with depot base loads but **no EV charging**. This identifies peak periods and the normal Step-2 network condition.

In [ ]:
zero = zero_schedule(base_p.index)
results = run_timeseries(network_factory, base_p, base_q, zero, bus_map)
summary = network_summary(results, "step2_depot_base_load")
critical_time = results.loc[results.max_line_loading_percent.idxmax(), "time"]
print("Critical normal-operation hour:", critical_time)
display(summary)


### What this cell does — Step-2 contingency and strict limits

Freezes the network at the critical Step-2 hour. N-1 now uses only the **all-closed** switch state, as required by the project update. The same outage set is evaluated once with normal limits and once with 0.95–1.05 pu voltage limits.

In [ ]:
critical_net = build_snapshot_network(network_factory, base_p, base_q, zero, bus_map, critical_time)
n1_normal, n1_normal_viol = run_n1(critical_net, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)
n1_strict, n1_strict_viol = run_n1(critical_net, v_min=config.V_MIN_STRICT, v_max=config.V_MAX_STRICT, switch_configs=SWITCH_CONFIGS_ALL_CLOSED)

print("Normal N-1 electrical limits OK:", int(n1_normal.electrical_limits_ok.sum()), "/", len(n1_normal))
print("Strict N-1 electrical limits OK:", int(n1_strict.electrical_limits_ok.sum()), "/", len(n1_strict))
display(n1_normal)


### What this cell does — Export Step 2

Saves the bus-screening evidence, 336-hour network results and contingency results. No charger-dependent `network_limits.csv` is created yet; that belongs to Step 3 after the charging infrastructure has been designed.

In [ ]:
out = config.RESULTS / "step2_timeseries"
out.mkdir(parents=True, exist_ok=True)
screening.to_csv(out / "connection_point_screening.csv", index=False)
results.to_csv(out / "network_timeseries.csv", index=False)
summary.to_csv(out / "network_summary.csv", index=False)
n1_normal.to_csv(out / "n1_normal_summary.csv", index=False)
n1_normal_viol.to_csv(out / "n1_normal_violations.csv", index=False)
n1_strict.to_csv(out / "n1_strict_summary.csv", index=False)
n1_strict_viol.to_csv(out / "n1_strict_violations.csv", index=False)
save_network_plot(results, out / "network_loading.png", "Step 2 — Depot base-load network")
print("Saved to", out)
